# Namakan AI - Fine-Tuning Pipeline
## Client: {{CLIENT_NAME}} | Version: {{VERSION}}

**⚠️ SECURITY: This Colab instance is dedicated to {{CLIENT_NAME}}. Do not use for any other client.**

This notebook:
1. Downloads training data from Google Drive
2. Prepares and validates data
3. Trains model using QLoRA
4. Merges and exports model
5. Uploads to client storage
6. Notifies Namakan via webhook

## Configuration

**⚠️ DO NOT SHARE THIS NOTEBOOK. Each client gets their own copy.**

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CONFIGURATION - Fill in before running
# ═══════════════════════════════════════════════════════════════

CLIENT_NAME = "{{CLIENT_NAME}}"  # e.g., "acme_corp"
CLIENT_SLUG = "{{CLIENT_SLUG}}"  # e.g., "acme"

# Training data from Drive
TRAINING_DATA_FILE_ID = "{{TRAINING_DATA_FILE_ID}}"  # CSV file ID

# Model configuration
BASE_MODEL = "Qwen/Qwen2.5-8B-Instruct"
OUTPUT_MODEL_NAME = "{{CLIENT_SLUG}}-v{{VERSION}}"

# LoRA configuration
LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
EPOCHS = 3
BATCH_SIZE = 4
LEARNING_RATE = 2e-4
MAX_SEQ_LENGTH = 2048

# Output: Drive folder where model will be saved
OUTPUT_FOLDER_ID = "{{MODEL_OUTPUT_FOLDER_ID}}"

# Webhook for notification
WEBHOOK_URL = "{{N8N_WEBHOOK_URL}}"

print(f"Client: {CLIENT_NAME}")
print(f"Model: {OUTPUT_MODEL_NAME}")
print(f"Base Model: {BASE_MODEL}")

## 1. Install Dependencies

In [ ]:
%pip install -q transformers>=4.36.0 accelerate>=0.25.0 peft>=0.7.0 bitsandbytes>=0.41.0 \
    datasets>=2.14.0 huggingface_hub>=0.19.0 pandas>=2.0.0 google-api-python-client>=2.100.0

In [ ]:
import os
import torch
from google.colab import drive
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset
import pandas as pd
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import io
import requests
import time

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Mount Google Drive

In [ ]:
drive.mount('/content/drive')
DRIVE_PATH = "/content/drive/MyDrive/Namakan"
print(f"Drive mounted at: {DRIVE_PATH}")

## 3. Download Training Data

In [ ]:
def download_file_from_drive(file_id, destination):
    """Download file from Google Drive given file ID"""
    service = build('drive', 'v3', developerKey='AIzaSyBWpT')
    
    try:
        request = service.files().get_media(fileId=file_id)
        fh = io.FileIO(destination, 'wb')
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            if status:
                print(f"Download {int(status.progress() * 100)}%")
        print(f"Downloaded to: {destination}")
        return True
    except Exception as e:
        print(f"Error downloading: {e}")
        return False

# Create client directory
CLIENT_DIR = f"{DRIVE_PATH}/{CLIENT_NAME}"
os.makedirs(CLIENT_DIR, exist_ok=True)

# Download training data
TRAINING_DATA_PATH = f"{CLIENT_DIR}/training_data.csv"
print(f"\nDownloading training data for {CLIENT_NAME}...")
success = download_file_from_drive(TRAINING_DATA_FILE_ID, TRAINING_DATA_PATH)

if success:
    df = pd.read_csv(TRAINING_DATA_PATH)
    print(f"\nLoaded {len(df)} training examples")
    print(f"Columns: {list(df.columns)}")
else:
    raise Exception("Failed to download training data")

## 4. Validate and Prepare Data

In [ ]:
# Security: Final PII check before training
import re

def final_pii_check(text):
    """Final check for any remaining PII"""
    if pd.isna(text):
        return text
    text = str(text)
    
    # Check for common PII patterns
    patterns = [
        r'\b[A-Z][a-z]+ [A-Z][a-z]+\b',  # Names
        r'\b\d{3}-\d{2}-\d{4}\b',      # SSN
        r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b',  # Phone
        r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b'  # Email
    ]
    
    for pattern in patterns:
        matches = re.findall(pattern, text)
        if matches:
            # Check if already redacted
            if '[REDACTED]' in text or '[PERSON]' in text:
                continue
            return f"[FLAGGED: {' '.join(matches[:3])]}]"
    
    return text

# Apply final check
print("Running final PII validation...")
flagged = 0
for col in df.columns:
    if df[col].dtype == 'object':
        original_count = len(df)
        df[col] = df[col].apply(final_pii_check)
        new_count = df[df[col].str.contains('FLAGGED', na=False)].shape[0]
        flagged += new_count

if flagged > 0:
    print(f"⚠️ WARNING: {flagged} potential PII items found and flagged")
else:
    print("✓ No PII detected - data is clean")

# Validate required columns
required_cols = ['instruction', 'input', 'output']
missing = [col for col in required_cols if col not in df.columns]
if missing:
    raise Exception(f"Missing required columns: {missing}")

print(f"✓ Data validated: {len(df)} examples, columns OK")

## 5. Load Base Model with QLoRA

In [ ]:
def load_model_and_tokenizer():
    """Load Qwen2.5-8B with 4-bit quantization"""
    
    print(f"Loading tokenizer from {BASE_MODEL}...")
    tokenizer = AutoTokenizer.from_pretrained(
        BASE_MODEL,
        trust_remote_code=True,
        padding_side="right"
    )
    tokenizer.pad_token = tokenizer.eos_token
    
    print("Loading model with 4-bit quantization...")
    
    # 4-bit quantization config
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )
    
    # Load model
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
    
    # Prepare for k-bit training
    model = prepare_model_for_kbit_training(model)
    
    # Configure LoRA
    lora_config = LoraConfig(
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj"
        ]
    )
    
    # Apply LoRA
    model = get_peft_model(model, lora_config)
    
    # Print trainable params
    model.print_trainable_parameters()
    
    return model, tokenizer

model, tokenizer = load_model_and_tokenizer()

## 6. Prepare Dataset

In [ ]:
def format_instruction_example(row):
    """Format example in Qwen instruction-following format"""
    instruction = str(row['instruction']) if pd.notna(row['instruction']) else ""
    input_text = str(row.get('input', '')) if pd.notna(row.get('input', '')) else ""
    output = str(row['output']) if pd.notna(row['output']) else ""
    
    # Qwen chat format
    if input_text:
        text = f"<|im_start|>user\n{instruction}\n{input_text}<|im_end|>\n<|im_start|>assistant\n{output}<|im_end|>"
    else:
        text = f"<|im_start|>user\n{instruction}<|im_end|>\n<|im_start|>assistant\n{output}<|im_end|>"
    
    return text

def tokenize_function(examples):
    """Tokenize texts"""
    result = tokenizer(
        examples['text'],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding="max_length"
    )
    result['labels'] = result['input_ids'].copy()
    return result

# Format all examples
print("Formatting training examples...")
texts = df.apply(format_instruction_example, axis=1).tolist()

# Create dataset
dataset = Dataset.from_dict({"text": texts})
print(f"Created dataset with {len(dataset)} examples")

# Tokenize
print("Tokenizing...")
dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset.column_names,
    desc="Tokenizing"
)

print(f"✓ Dataset ready: {len(dataset)} examples, {MAX_SEQ_LENGTH} max length")

## 7. Train Model

In [ ]:
def train_model(model, dataset, tokenizer):
    """Fine-tune with QLoRA"""
    
    # Data collator
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False  # Causal LM
    )
    
    # Output directory
    output_dir = f"/content/{OUTPUT_MODEL_NAME}"
    
    # Training arguments
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=4,
        learning_rate=LEARNING_RATE,
        warmup_steps=100,
        logging_steps=10,
        save_steps=500,
        fp16=True,
        report_to="none",
        optim="paged_adamw_8bit",
        max_grad_norm=0.3,
        save_strategy="no"  # Don't save checkpoints
    )
    
    # Create trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset,
        data_collator=data_collator,
    )
    
    print(f"\n{'='*60}")
    print(f"Starting training for {EPOCHS} epochs...")
    print(f"Training examples: {len(dataset)}")
    print(f"Batch size: {BATCH_SIZE} (with gradient accumulation: {BATCH_SIZE * 4})")
    print(f"{'='*60}\n")
    
    start_time = time.time()
    trainer.train()
    training_time = time.time() - start_time
    
    print(f"\n✓ Training complete in {training_time/60:.1f} minutes")
    
    return model, training_time

model, training_time = train_model(model, dataset, tokenizer)

## 8. Save and Export Model

In [ ]:
def save_and_export_model(model, tokenizer):
    """Merge LoRA weights and save"""
    
    output_dir = f"/content/{OUTPUT_MODEL_NAME}"
    
    print("Merging LoRA weights...")
    merged_model = model.merge_and_unload()
    
    print(f"Saving to {output_dir}...")
    merged_model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    
    # Create Modelfile for Ollama
    modelfile_content = f"""FROM ./{OUTPUT_MODEL_NAME}
PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER top_k 40
PARAMETER num_keep 24
TEMPLATE \"\"\"<|im_start|>user\n{{ .Prompt }}<|im_end|>\n<|im_start|>assistant\n{{ .Response }}<|im_end|>\"\"\"
SYSTEM \"You are an AI assistant for {CLIENT_NAME}. Use the context from your training to provide accurate, helpful responses.\"
"""
    
    with open(f"{output_dir}/Modelfile", "w") as f:
        f.write(modelfile_content)
    
    print(f"✓ Model saved with Modelfile for Ollama")
    
    return output_dir

model_dir = save_and_export_model(model, tokenizer)

## 9. Upload to Google Drive

In [ ]:
def upload_folder_to_drive(local_folder, drive_folder_id):
    """Upload model folder to Google Drive"""
    from googleapiclient.http import MediaFileUpload
    
    service = build('drive', 'v3', developerKey='AIzaSyBWpT')
    
    def upload_file(file_path, parent_id):
        file_name = os.path.basename(file_path)
        mime_type = 'text/plain'
        if file_name.endswith('.json'):
            mime_type = 'application/json'
        elif file_name.endswith('.bin') or file_name.endswith('.safetensors'):
            mime_type = 'application/octet-stream'
        
        file_metadata = {
            'name': file_name,
            'parents': [parent_id]
        }
        
        media = MediaFileUpload(file_path, mimetype=mime_type, resumable=True)
        
        try:
            uploaded = service.files().create(
                body=file_metadata,
                media_body=media,
                fields='id'
            ).execute()
            print(f"  ✓ Uploaded: {file_name}")
            return uploaded.get('id')
        except Exception as e:
            print(f"  ✗ Error uploading {file_name}: {e}")
            return None
    
    # Create client folder in Drive
    client_folder_metadata = {
        'name': f'{CLIENT_NAME}/models',
        'mimeType': 'application/vnd.google-apps.folder',
        'parents': [drive_folder_id]
    }
    
    try:
        client_folder = service.files().create(
            body=client_folder_metadata,
            fields='id'
        ).execute()
        client_folder_id = client_folder.get('id')
        print(f"Created folder: {CLIENT_NAME}/models")
    except:
        # Folder might already exist, search for it
        results = service.files().list(
            q=f"name='{CLIENT_NAME}' and mimeType='application/vnd.google-apps.folder'",
            spaces='drive'
        ).execute()
        if results.get('files'):
            client_folder_id = results['files'][0]['id']
            print(f"Using existing folder: {CLIENT_NAME}")
        else:
            raise Exception("Could not create or find folder")
    
    # Upload all files recursively
    print(f"\nUploading model files...")
    for root, dirs, files in os.walk(local_folder):
        for file in files:
            file_path = os.path.join(root, file)
            upload_file(file_path, client_folder_id)
    
    return f"https://drive.google.com/drive/folders/{client_folder_id}"

print("Uploading model to Google Drive...")
drive_link = upload_folder_to_drive(model_dir, OUTPUT_FOLDER_ID)
print(f"\n✓ Model uploaded: {drive_link}")

## 10. Notify Completion

In [ ]:
def notify_completion(success=True, error=None):
    """Send webhook notification"""
    
    if success:
        payload = {
            "client": CLIENT_NAME,
            "model": OUTPUT_MODEL_NAME,
            "status": "complete",
            "training_examples": len(dataset),
            "training_time_minutes": int(training_time / 60),
            "drive_link": drive_link,
            "colab_instance": os.environ.get('COLAB_NAME', 'unknown')
        }
        
        print("Sending success notification...")
        print(f"Payload: {payload}")
    else:
        payload = {
            "client": CLIENT_NAME,
            "model": OUTPUT_MODEL_NAME,
            "status": "failed",
            "error": str(error)
        }
        print(f"Sending failure notification: {error}")
    
    try:
        response = requests.post(WEBHOOK_URL, json=payload, timeout=30)
        print(f"Webhook response: {response.status_code}")
        return response.status_code == 200
    except Exception as e:
        print(f"Webhook failed: {e}")
        return False

# Send notification
notify_completion(success=True)

print("\n" + "="*60)
print("✓ FINE-TUNING COMPLETE")
print("="*60)
print(f"Client: {CLIENT_NAME}")
print(f"Model: {OUTPUT_MODEL_NAME}")
print(f"Training time: {training_time/60:.1f} minutes")
print(f"Model location: {drive_link}")
print("="*60)

---

## Security Notes

1. **This Colab instance is isolated** - Per-client instances prevent data mixing
2. **No data retention** - Model files are uploaded to Drive and local Colab instance is deleted when runtime ends
3. **PII check** - Final validation before training ensures no personal data in training
4. **Credentials not stored** - File IDs and webhook URLs are temporary
5. **No internet access during training** - Colab runs offline once data is loaded

## Troubleshooting

- **OOM (Out of Memory)**: Reduce BATCH_SIZE to 2 or 1
- **Slow training**: Normal on T4 GPU, ~45 minutes expected
- **Upload fails**: Check Drive permissions and folder ID
- **Webhook fails**: N8N instance may be down, check manually